Reproducing Sawyer et al. 2023 (doi:10.1029/2023GL104733) Figure 2a–c. THEMIS-B periselene ~15 km, 2013-12-04 00:05–00:16 UT. THEMIS-C upstream monitor. Target: THB shows crustal-field departure from THC from ~00:08.

In [ ]:
import pyspedas
from pyspedas import tplot, tplot_names, tlimit

trange = ['2013-12-04/00:00:00', '2013-12-04/00:30:00']
tlimit(['2013-12-04/00:05:00', '2013-12-04/00:16:00']) # set time limit for event instead of full 24h

# FGM for both probes: THB = near-surface, THC = upstream monitor
pyspedas.projects.themis.fgm(probe=['b', 'c'], trange=trange, level='l2')

# State data: position -> altitude
pyspedas.projects.themis.state(probe='b', trange=trange)

# tplot_names()   # what actually loaded

In [ ]:
tplot(['thb_fgs_gse', 'thc_fgs_gse'], save_pdf='../figures/sawyer_fgm_event')

Availability check:

In [ ]:
from pyspedas.projects.themis import esa

trange = ['2013-12-04/00:05:00', '2013-12-04/00:16:00']
esa(trange=trange, probe='b', level='l2') # ESA — ion and electron moments
tplot_names()

Full (peif/peef), reduced (peir/peer) and burst (peib/peeb) modes are all loaded correctly. HT transform feasible. Quick coverage check:

In [ ]:
from pyspedas import get_data
import numpy as np

for v in ['thb_peir_velocity_dsl', 'thb_peer_velocity_dsl',
          'thb_peib_velocity_dsl', 'thb_peeb_velocity_dsl',
          'thb_peir_density', 'thb_peer_density']:
    d = get_data(v)
    if d is None or len(d.times) == 0:
        print(f'{v}: EMPTY')
    else:
        dt = np.median(np.diff(d.times))
        print(f'{v}: {len(d.times)} pts, median cadence {dt:.1f} s')

Results: cadence is 4.3 across the board (ARTEMIS spin period for this event). Sawyer reported approx 4s, we obtain 4.3s-- agreement!

Next, check solar wind flags:

In [ ]:
import numpy as np
from pyspedas import time_double

t0, t1 = time_double('2013-12-04/00:05:00'), time_double('2013-12-04/00:16:00')

for v in ['thb_iesa_solarwind_flag', 'thb_eesa_solarwind_flag']:
    d = get_data(v)
    m = (d.times >= t0) & (d.times <= t1)
    print(v, np.unique(d.y[m]))

for v in ['thb_peir_density', 'thb_peer_density']:
    d = get_data(v)
    m = (d.times >= t0) & (d.times <= t1)
    print(f'{v}: {m.sum()} pts in window, '
          f'median {np.nanmedian(d.y[m]):.2f} cm^-3')


Results: ESA was not running in solar wind mode, i.e. sensor runs in magnetospheric mode with accepted saturation (as expected). Densities agree to 2%, both sitting on nominal 5 cm^-3. Check velocities:

In [ ]:
for v in ['thb_peir_velocity_gse', 'thb_peer_velocity_gse']:
    d = get_data(v)
    m = (d.times >= t0) & (d.times <= t1)
    print(v, np.nanmedian(d.y[m], axis=0))

Interesting! Why nan? Let's diagnose:

In [ ]:
d = get_data('thb_peer_velocity_gse')
m = (d.times >= t0) & (d.times <= t1)
print('finite fraction:', np.isfinite(d.y[m]).mean())

# is it the whole day or just the window?
print('finite fraction, all data:', np.isfinite(d.y).mean())

for v in ['thb_peer_sc_pot', 'thb_peer_data_quality']:
    dd = get_data(v)
    mm = (dd.times >= t0) & (dd.times <= t1)
    print(v, np.unique(dd.y[mm]))

Where are my electron velocities??

In [ ]:
# does full mode have what reduced doesn't?
for v in ['thb_peef_velocity_gse', 'thb_peeb_velocity_gse']:
    d = get_data(v)
    m = (d.times >= t0) & (d.times <= t1)
    print(v, np.isfinite(d.y[m]).mean(), np.nanmedian(d.y[m], axis=0))

# where does data_quality == 2 fall?
d = get_data('thb_peer_data_quality')
m = (d.times >= t0) & (d.times <= t1)
print('fraction flagged:', (d.y[m] != 0).mean())

Yes! Full and burst modes are finite and agree. Interpretation: reduced-mode electron moments were suppressed by the pipeline over whole window, peef and peeb come through a different processing path and are unaffected.

In [ ]:
for v in ['thb_peef_velocity_gse', 'thb_peif_velocity_gse']:
    d = get_data(v)
    m = (d.times >= t0) & (d.times <= t1)
    print(v, m.sum(), 'pts,',
          f'{np.median(np.diff(d.times[m])):.1f} s')

Lit Review figure:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pyspedas

try:
    from pytplot import get_data
except ImportError:
    from pyspedas import get_data

# ---------------------------------------------------------------- settings
TRANGE = ['2013-12-04/00:05', '2013-12-04/00:16']

# Crossing interval: PLACEHOLDER, replace with Sawyer 2023 Fig. 2 times
CROSS = ('2013-12-04 00:11:00', '2013-12-04 00:11:20')

# Optional zoom window for the plot (None = whole TRANGE)
ZOOM = ('2013-12-04 00:09:00', '2013-12-04 00:14:00')

OUT = '../figures/sawyer_instrumentation'

# Particle panels to show
SPECIES = ('e',)

# Slide-5 palette: navy for instruments, orange for the physics/event
NAVY, ORANGE, GREY = '#1b3a5c', '#d84315', '#9e9e9e'
COMP = {'x': '#1f4e9c', 'y': '#2e7d32', 'z': '#c62828'}  # same as tplot B/G/R

# ---------------------------------------------------------------- load
pyspedas.projects.themis.fgm(probe='b', trange=TRANGE, level='l2')
pyspedas.projects.themis.esa(probe='b', trange=TRANGE, level='l2')


def to_dt64(t):
    return (np.asarray(t, dtype='float64') * 1e9).astype('datetime64[ns]')


def load(name):
    d = get_data(name)
    if d is None:
        return None
    t, y = to_dt64(d.times), np.asarray(d.y, dtype=float)
    if np.all(np.isnan(y)):
        return None
    return t, y


def cadence_s(t):
    return float(np.median(np.diff(t)) / np.timedelta64(1, 's'))


# Prefer high-rate field data; fall back to spin-resolution
B, b_name = None, None
for name in ('thb_fgh_gse', 'thb_fgl_gse', 'thb_fgs_gse'):
    B = load(name)
    if B is not None:
        b_name = name
        break
if B is None:
    raise RuntimeError('No THB FGM data loaded')

Vi = load('thb_peir_velocity_gse') if 'i' in SPECIES else None
Ve = load('thb_peef_velocity_gse') if 'e' in SPECIES else None  # peer is NaN here

c0, c1 = (np.datetime64(s.replace(' ', 'T')) for s in CROSS)

# ---------------------------------------------------------------- plot
panels = [p for p in (Vi, Ve) if p is not None]
fig, axes = plt.subplots(1 + len(panels), 1, figsize=(6.4, 2.1 * (1 + len(panels))),
                         sharex=True, constrained_layout=True)
axes = np.atleast_1d(axes)

# Panel 1: magnetic field
ax = axes[0]
tB, yB = B
for i, c in enumerate('xyz'):
    ax.plot(tB, yB[:, i], lw=0.8, color=COMP[c], label=f'$B_{c}$')
ax.plot(tB, np.linalg.norm(yB, axis=1), lw=1.0, color='k', label='$|B|$')
ax.set_ylabel('B [nT, GSE]')
ax.legend(loc='upper left', ncol=4, fontsize=8, frameon=False)
ax.set_title(f'{b_name.split("_")[1].upper()} magnetometer: {1 / cadence_s(tB):.0f} samples/s',
             loc='right', fontsize=9, color=NAVY)


def particle_panel(ax, data, species):
    t, y = data
    for i, c in enumerate('xyz'):
        ax.plot(t, y[:, i], '-o', lw=0.8, ms=3, color=COMP[c], label=f'$V_{{{species},{c}}}$')
    n_in = int(np.sum((t >= c0) & (t <= c1)))
    ax.set_ylabel(f'$V_{species}$ [km/s, GSE]')
    ax.legend(loc='upper left', ncol=3, fontsize=8, frameon=False)
    ax.set_title(f'ESA: one sample per {cadence_s(t):.1f} s, {n_in} inside crossing',
                 loc='right', fontsize=9, color=NAVY)
    return n_in


counts = {}
for ax, (data, sp) in zip(axes[1:], [(p, s) for p, s in ((Vi, 'i'), (Ve, 'e')) if p is not None]):
    counts[sp] = particle_panel(ax, data, sp)

# Shade the crossing on every panel
for ax in axes:
    ax.axvspan(c0, c1, color=ORANGE, alpha=0.15, lw=0)
axes[0].text(c0 + (c1 - c0) / 2, 0.02, 'crossing', transform=axes[0].get_xaxis_transform(),
             ha='center', va='bottom', fontsize=8, color=ORANGE)

# Time axis
if ZOOM:
    axes[-1].set_xlim(*(np.datetime64(s.replace(' ', 'T')) for s in ZOOM))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
axes[-1].set_xlabel('UT, 2013-12-04')
for ax in axes:
    ax.tick_params(labelsize=9)
    ax.spines[['top', 'right']].set_visible(False)

fig.savefig(OUT + '.pdf')
fig.savefig(OUT + '.png', dpi=300)
print(f'Field product: {b_name} ({cadence_s(tB):.3g} s)')
for sp, n in counts.items():
    print(f'{sp}: {n} ESA samples inside {CROSS[0]} -> {CROSS[1]}')